In [3]:
import re
import pandas as pd
from astropy.io import fits
from astropy.wcs import WCS

# ============================================================
# INPUTS
# ============================================================
reg_file = "/Users/aishwarya/Desktop/source_cdfs.reg"
fits_file = "/Users/aishwarya/Desktop/new_cdfs/trimmed_2deg/trim2deg_y_sci.fits"
output_cat = "/Users/aishwarya/Desktop/source_cdfs_RADEC.cat"

# ============================================================
# LOAD WCS
# ============================================================
hdu = fits.open(fits_file)
wcs = WCS(hdu[0].header)

# ============================================================
# READ REG FILE (PIXEL COORDS)
# ============================================================
x_list = []
y_list = []

with open(reg_file, 'r') as f:
    lines = f.readlines()

for line in lines:
    line = line.strip()

    # Skip headers
    if line.startswith('#') or line.startswith('global') or line.startswith('physical') or line == '':
        continue

    if line.startswith('circle'):
        match = re.search(r'circle\(([^,]+),([^,]+),', line)
        if match:
            x = float(match.group(1))
            y = float(match.group(2))

            x_list.append(x)
            y_list.append(y)

# ============================================================
# CONVERT PIXEL → RA/DEC
# ============================================================
ra, dec = wcs.pixel_to_world_values(x_list, y_list)

# ============================================================
# SAVE OUTPUT
# ============================================================
df = pd.DataFrame({
    'ALPHA_J2000': ra,
    'DELTA_J2000': dec
})

df.to_csv(output_cat, index=False, sep=' ')

print(f"Saved: {output_cat}")
print(df.head())

Saved: /Users/aishwarya/Desktop/source_cdfs_RADEC.cat
   ALPHA_J2000  DELTA_J2000
0    53.035756   -29.013265
1    52.473055   -28.576205
2    51.988985   -28.545289
3    51.883089   -28.540280
4    52.087312   -28.535903


In [5]:
import pandas as pd

# ============================================================
# INPUT / OUTPUT
# ============================================================
cat_file = "/Users/aishwarya/Desktop/source_cdfs_clean.cat"
output_reg = "/Users/aishwarya/Desktop/source_cdfs_points.reg"

# ============================================================
# LOAD CATALOG
# ============================================================
df = pd.read_csv(cat_file, delim_whitespace=True, comment='#')

# If column names are missing, assign manually
if df.shape[1] >= 2:
    df.columns = ['ALPHA_J2000', 'DELTA_J2000'] + list(df.columns[2:])

# ============================================================
# WRITE DS9 REGION FILE (POINTS)
# ============================================================
with open(output_reg, 'w') as f:
    f.write("# Region file format: DS9 version 4.1\n")
    f.write("global color=green width=2\n")
    f.write("fk5\n")

    for _, row in df.iterrows():
        ra = row['ALPHA_J2000']
        dec = row['DELTA_J2000']

        f.write(f"point({ra},{dec}) # point=cross 5 width=1\n")

print(f"Saved region file: {output_reg}")

Saved region file: /Users/aishwarya/Desktop/source_cdfs_points.reg


/var/folders/9v/db3j63px68q3f33kdb7_7wbc0000gn/T/ipykernel_11634/80745633.py:12: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  df = pd.read_csv(cat_file, delim_whitespace=True, comment='#')


In [12]:
import pandas as pd

# Input and output files
txt_file = "/Users/aishwarya/Desktop/comparision/Source.txt"
csv_file = "/Users/aishwarya/Desktop/CSV/source.csv"

# Read the txt file
df = pd.read_csv(txt_file, delim_whitespace=True, comment='#', header=None)

# Assign existing column names
df.columns = ["ra", "dec"]

# Add empty third column
df["source(yes/no)"] = ""

# Save as CSV
df.to_csv(csv_file, index=False)

print("Conversion complete!")

Conversion complete!


/var/folders/9v/db3j63px68q3f33kdb7_7wbc0000gn/T/ipykernel_11634/3621472433.py:8: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  df = pd.read_csv(txt_file, delim_whitespace=True, comment='#', header=None)
